## 1. Метод квазилинеаризации Ньютона

Нам дана краевая задача:
$$y'' = \frac{y -y^3} {\varepsilon}$$
$$y(0) = A, \quad y(1) = B$$
### Аппроксимация
Введем равномерную сетку $x_j = j h$, где $h = 1/N$. Вторую производную аппроксимируем центральной разностью:
$$ \frac{y_{j-1} - 2y_j + y_{j+1}}{h^2}=f(y_j)$$

### Линеаризация
$$f'(y)=\left(\frac{y -y^3} {\varepsilon}\right)'=\frac{1 -3y^2} {\varepsilon}$$


В итоге

$$ \frac{y_{j-1}^{k+1} - 2y_j^{k+1} + y_{j+1}^{k+1}}{h^2}=\frac{y_j^k -(y_j^k)^3} {\varepsilon}+\frac{1 -3(y_j^k)^2} {\varepsilon}(y_j^{k+1}-y_j^k)$$
или

$$\frac{\varepsilon}{h^2} y_{j-1}^{k+1} + \left( -\frac{2\varepsilon}{h^2} - 1 + 3(y_j^k)^2 \right) y_j^{k+1} + \frac{\varepsilon}{h^2} y_{j+1}^{k+1} = 2(y_j^k)^3$$

## 2. Начальное приближения

Аналитически локализованный пик для уравнения $\epsilon y'' = y - y^3$ описывается через гиперболический секанс: $y_{peak}(x) = \frac{\sqrt{2}}{\cosh(x/\sqrt{\epsilon})}$. Пограничные слои, в свою очередь, затухают по экспоненте.

Для $n=2$ (один внутренний пик в точке $x = 1/2$):$$y^{(0)}(x) = A e^{-x/\sqrt{\epsilon}} + B e^{-(1-x)/\sqrt{\epsilon}} + \frac{\sqrt{2}}{\cosh\left(\frac{x - 0.5}{\sqrt{\epsilon}}\right)}$$Для $n=3$ (два внутренних пика в точках $x = 1/3$ и $x = 2/3$):$$y^{(0)}(x) = A e^{-x/\sqrt{\epsilon}} + B e^{-(1-x)/\sqrt{\epsilon}} + \frac{\sqrt{2}}{\cosh\left(\frac{x - 1/3}{\sqrt{\epsilon}}\right)} + \frac{\sqrt{2}}{\cosh\left(\frac{x - 2/3}{\sqrt{\epsilon}}\right)}$$

Выражени для пиков получаем из условия 
$$\lim_{\epsilon \to 0} y(x,\epsilon)=\sqrt{2}$$

Поскольку уравнение симметрично, перед каждым слагаемым пика можно поставить минус — именно комбинируя знаки пиков, можно получить те самые разные линейно независимые решения, о которых говорится в условии

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

def solve_spike_bvp(eps, n, A, B, N, max_iter=100, tol=1e-6):
    h = 1.0 / N
    x = np.linspace(0, 1, N + 1)
    
    # Формируем начальное приближение
    
    # Задаем пограничные слои, которые экспоненциально спадают к нулю от краев
    y = A * np.exp(-x / np.sqrt(eps)) + B * np.exp(-(1 - x) / np.sqrt(eps))
    
    for i in range(1, n):
        x_i = i / n
        spike = np.sqrt(2) / np.cosh((x - x_i) / np.sqrt(2*eps))
        # Складываем слои и пики 
        y = np.maximum(y, spike) 
        
    y[0] = A
    y[-1] = B
    # Итерации Ньютона
    for it in range(max_iter):
        y_k = y[1:-1]
        
        # Главная и побочные диагонали матрицы A
        main_diag = 3 * y_k**2 - 1 - 2 * eps / h**2
        off_diag = (eps / h**2) * np.ones(N - 2)
        A_mat = diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csr')
        
        # правая часть: 2 * (y^k)^3
        RHS = 2 * y_k**3
        
        # Учет краевых условий (переносим известные значения с краев сетки вправо)
        RHS[0] -= (eps / h**2) * A
        RHS[-1] -= (eps / h**2) * B
        
        # Решаем систему A * y^{k+1} = RHS
        y_next = spsolve(A_mat, RHS)
        
        # Оцениваем разницу для критерия остановки
        dy_norm = np.linalg.norm(y_next - y_k, np.inf)
        
        # Обновляем значения
        y[1:-1] = y_next
        
        if dy_norm < tol:
            print(f"[eps={eps}, n={n}] Сошлось за {it+1} итераций.")
            break
    else:
        print(f"[eps={eps}, n={n}] ВНИМАНИЕ: Не сошлось за {max_iter} итераций! Ошибка: {dy_norm:.2e}")
        
    return x, y

In [ ]:
eps_values = [1e-2, 1e-3, 1e-4]

A_val, B_val =1, 1

plt.figure(figsize=(14, 6))


plt.subplot(1, 2, 1)
plt.title("Пиковые структуры для $n=2$")
for eps in eps_values:
    # Для eps=1e-4 сетку лучше сделать гуще, чтобы поймать очень узкий пик
    N_grid = 5000 if eps == 1e-4 else 2000 
    x, y = solve_spike_bvp(eps, 2, A_val, B_val, N_grid)
    plt.plot(x, y, label=f'$\epsilon = 10^{{{int(np.log10(eps))}}}$')

plt.axhline(np.sqrt(2), color='gray', linestyle='--', alpha=0.5, label='$\lim = \sqrt{2}$')
plt.xlabel('$x$')
plt.ylabel('$y(x)$')
plt.grid(True, linestyle=':')
plt.legend()

# --- Случай n = 3 ---
plt.subplot(1, 2, 2)
plt.title("Пиковые структуры для $n=3$")
for eps in eps_values:
    N_grid = 5000 if eps == 1e-4 else 2000
    x, y = solve_spike_bvp(eps, 3, A_val, B_val, N_grid)
    plt.plot(x, y, label=f'$\epsilon = 10^{{{int(np.log10(eps))}}}$')

plt.axhline(np.sqrt(2), color='gray', linestyle='--', alpha=0.5)
plt.xlabel('$x$')
plt.ylabel('$y(x)$')
plt.grid(True, linestyle=':')
plt.legend()

plt.tight_layout()
plt.show()